In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType,DecimalType
from pyspark.sql import functions  as F
from delta  import DeltaTable 

In [0]:
# disabling the spark UI 
# spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
# schema creation
_expected_schema = StructType(
    [
        StructField("hvfhs_license_num", StringType(), True),
        StructField("dispatching_base_nu", StringType(), True),
        StructField("originating_base_num", StringType(), True),
        StructField("request_datetime", TimestampType(), True),
        StructField("on_scene_datetime", TimestampType(), True),
        StructField("pickup_datetime", TimestampType(), True),
        StructField("dropoff_datetime", TimestampType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("trip_miles", DecimalType(), True),
        StructField("trip_time", IntegerType(), True),
        StructField("base_passenger_fare", DecimalType(), True),
        StructField("tolls", DecimalType(), True),
        StructField("bcf", DecimalType(), True),
        StructField("sales_tax", DecimalType(), True),
        StructField("congestion_surcharge", DecimalType(), True),
        StructField("airport_fee", DecimalType(), True),
        StructField("tips", DecimalType(), True),
        StructField("driver_pay", DecimalType(), True),
        StructField("shared_request_flag", DecimalType(), True),
        StructField("shared_match_flag", StringType(), True),
        StructField("access_a_ride_flag", StringType(), True),
        StructField("wav_request_flag", StringType(), True),
        StructField("wav_match_flag", StringType(), True),
        StructField("cbd_congestion_fee", DecimalType(), True),
    ]
)

In [0]:
%run ../CatalogandTableCreation/TableandCatlogCreation

In [0]:
df = (
           spark.read.format('parquet').load('/Volumes/nyctaxi/inbound/inbounddata/*.parquet')
     )
df.createOrReplaceTempView('inboundDataframeTview')

In [0]:
# keeping the Loading Process Structred streaiming
inboundTableObject =  DeltaTable.forName(spark ,'nyctaxi.inbound.nyc_taxi_inbound')
# creation of the table object 
required_columns = [ i for i in df.schema.fieldNames() if i not in["rowId", "pickup_date", "createdTimestamp"]]

inboundTableDF = inboundTableObject.toDF()
destinationFieldnames = [i for i in inboundTableDF.schema.fieldNames() if i not in ["rowId", "pickup_date", "createdTimestamp"] 
]
insertClauseColumnNames ={ dest  : col(f"source.{source_columns}") for dest , source_columns in  zip(required_columns  , destinationFieldnames) }

columnString   = ','.join(required_columns)

# inserting the data into the  inbound layer table 

spark.sql(f""" INSERT INTO  nyctaxi.inbound.nyc_taxi_inbound
          (
              
             {columnString}

          )
          
          SELECT 
          {columnString}
          from 
          inboundDataframeTview
          """
)

In [0]:
# Moving the file to archive folder
files_list  = dbutils.fs.ls ("/Volumes/nyctaxi/inbound/inbounddata/")
for i in files_list:
    if i.path.endswith("parquet"):
        dbutils.fs.mv(i.path, "/Volumes/nyctaxi/inbound/inbounddata/Archive/")

In [0]:
# performanceAntipatterns 
# table object after the data load 
taxiInboundTableObject =  DeltaTable.forName(spark ,'nyctaxi.inbound.nyc_taxi_inbound')
taxiInboundTableDF = taxiInboundTableObject.toDF()

# # checking the data quality 
joinResult = ( taxiInboundTableDF.groupBy(
    'hvfhs_license_num').agg(F.sum("trip_miles")).alias('t1')
    .join(
         taxiInboundTableDF.groupBy(
    'hvfhs_license_num').agg(F.sum("base_passenger_fare")).alias('t2'),
         on=col("t1.hvfhs_license_num") == col("t2.hvfhs_license_num"),
         how="inner")
    )

joinResult.write.format("noop").mode("overwrite").save()